In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import current_timestamp, col




# --- 1. Define  Schema---
railway_schema = StructType([
    StructField("date", DateType(), True),
    StructField("station_no", IntegerType(), True),
    StructField("station_name", StringType(), True),
    StructField("delay", IntegerType(), True),
    StructField("train_no", IntegerType(), True)
])

# --- 2. Configure Paths ---
raw_path = "/Volumes/railway_analytics/bronze/raw_landing/train_delays.csv"
bronze_table_name = "railway_analytics.bronze.raw_train_delays"

df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .schema(railway_schema)
          .load(raw_path)
          .select("*", col("_metadata.file_path").alias("source_file")) 
)

# --- 3. Add Ingestion Timestamp ---
df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

((df_bronze.write
 .format("delta")
 .mode("append") 
 .saveAsTable(bronze_table_name)))

print(f"Bronze layer updated successfully in Unity Catalog: {bronze_table_name}")


Bronze layer updated successfully in Unity Catalog: railway_analytics.bronze.raw_train_delays


In [0]:
%sql
SELECT count(*) AS total_rows 
FROM railway_analytics.bronze.raw_train_delays;

total_rows
76857406
